# Baseline: BiLSTM + Attention on MediaPipe Pose Sequences

Orchestration notebook only -- all logic lives in `src/`. Run order:

1. Load labels, confirm participant split
2. Extract MediaPipe world landmarks (cached to `data/interim/`)
3. **Sanity baseline**: hand-crafted features (peak velocity, joint-angle ROM) + XGBoost -- catches broken labels/pipeline bugs cheaply before the deep model (addendum #5)
4. Leave-participant-out group k-fold CV on the sanity baseline (addendum #3) -- trustworthy accuracy estimate given only 9 participants
5. **Deep baseline**: full feature vectors (xyz + bone + velocity + visibility) + BiLSTM/attention, trained with augmented train split
6. Evaluate on locked val/test split, with TTA at inference (addendum #6)

See `action_classifier_protocol.md` sections 1-9 for full methodology.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if (Path.cwd().name == "baseline") else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

from src.config import (
    ACTIONS, PROCESSED_DIR, RUNS_DIR, SEED, SEQ_LEN, TEST_PARTICIPANTS,
    TRAIN_PARTICIPANTS, VAL_PARTICIPANTS,
)

np.random.seed(SEED)
torch.manual_seed(SEED)
print("project root:", PROJECT_ROOT)
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

project root: c:\Data_Tekken
torch: 2.5.1+cpu | cuda available: False


## 1. Load labels + confirm split

In [2]:
from src.data.dataset import load_labels, split_dataframe

df = load_labels()
train_df, val_df, test_df = split_dataframe(df)

print(f"total clips: {len(df)}")
print(df.groupby("action").size())
print()
print(f"train: {len(train_df)} clips (participants {TRAIN_PARTICIPANTS}, side+front)")
print(f"val:   {len(val_df)} clips (participant {VAL_PARTICIPANTS}, side-only)")
print(f"test:  {len(test_df)} clips (participants {TEST_PARTICIPANTS}, side-only)")
assert set(train_df.participant_id) & set(val_df.participant_id) == set()
assert set(train_df.participant_id) & set(test_df.participant_id) == set()
assert set(val_df.participant_id) & set(test_df.participant_id) == set()
print("\nno participant leakage across splits -- confirmed")

total clips: 642
action
kicking     210
punching    220
shooting    212
dtype: int64

train: 529 clips (participants [4, 5, 6, 7, 8, 9], side+front)
val:   30 clips (participant [1], side-only)
test:  49 clips (participants [2, 3], side-only)

no participant leakage across splits -- confirmed


## 2. Extract MediaPipe landmarks (cached)

First run is slow (~642 clips through MediaPipe Pose on CPU). Re-runs are instant --
everything lands in `data/interim/<action>/<clip>.npz` and is skipped if already there.

In [3]:
from tqdm.auto import tqdm
from src.data.dataset import _raw_landmarks

all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
for _, row in tqdm(all_df.iterrows(), total=len(all_df), desc="extracting landmarks"):
    _raw_landmarks(row)  # writes to data/interim/, no-ops if cached

print("landmark extraction complete / verified cached")

extracting landmarks:   0%|          | 0/608 [00:00<?, ?it/s]

landmark extraction complete / verified cached


## 3. Sanity baseline -- hand-crafted features + XGBoost

Peak wrist/ankle velocity + joint-angle ROM (elbow/knee/hip) per clip. Fast to build,
fast to train -- if this baseline can't beat chance (33%) by a wide margin, the labels
or extraction pipeline have a bug worth finding before touching the deep model.

In [4]:
from src.data.preprocess import preprocess_clip
from src.data.dataset import _raw_landmarks
from src.features.handcrafted import extract_handcrafted_features
from src.config import ACTION_TO_IDX

def handcrafted_matrix(frame_df):
    X, y, groups = [], [], []
    for _, row in tqdm(frame_df.iterrows(), total=len(frame_df), desc="handcrafted feats", leave=False):
        raw = _raw_landmarks(row)
        sample = preprocess_clip(raw["xyz"], raw["visibility"])  # deterministic, no aug
        X.append(extract_handcrafted_features(sample["xyz"]))
        y.append(ACTION_TO_IDX[row["action"]])
        groups.append(row["participant_id"])
    return np.stack(X), np.array(y), np.array(groups)

X_train_hc, y_train_hc, _ = handcrafted_matrix(train_df)
X_val_hc, y_val_hc, _ = handcrafted_matrix(val_df)
X_test_hc, y_test_hc, _ = handcrafted_matrix(test_df)
print(X_train_hc.shape, X_val_hc.shape, X_test_hc.shape)

handcrafted feats:   0%|          | 0/529 [00:00<?, ?it/s]

handcrafted feats:   0%|          | 0/30 [00:00<?, ?it/s]

handcrafted feats:   0%|          | 0/49 [00:00<?, ?it/s]

(529, 14) (30, 14) (49, 14)


In [5]:
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix

sanity_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    objective="multi:softmax", num_class=3, random_state=SEED,
)
sanity_model.fit(X_train_hc, y_train_hc)

for name, X, y in [("val", X_val_hc, y_val_hc), ("test", X_test_hc, y_test_hc)]:
    preds = sanity_model.predict(X)
    print(f"=== sanity baseline -- {name} ===")
    print(classification_report(y, preds, target_names=ACTIONS))
    print(confusion_matrix(y, preds))
    print()

=== sanity baseline -- val ===
              precision    recall  f1-score   support

     kicking       1.00      0.90      0.95        10
    punching       1.00      1.00      1.00        11
    shooting       0.90      1.00      0.95         9

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.96        30
weighted avg       0.97      0.97      0.97        30

[[ 9  0  1]
 [ 0 11  0]
 [ 0  0  9]]

=== sanity baseline -- test ===
              precision    recall  f1-score   support

     kicking       1.00      1.00      1.00        16
    punching       1.00      0.80      0.89        15
    shooting       0.86      1.00      0.92        18

    accuracy                           0.94        49
   macro avg       0.95      0.93      0.94        49
weighted avg       0.95      0.94      0.94        49

[[16  0  0]
 [ 0 12  3]
 [ 0  0 18]]



## 4. Leave-participant-out group k-fold CV (addendum #3)

Single held-out split (2-3 people) is a noisy estimate with only 9 participants.
Cross-validate the fast hand-crafted+XGBoost baseline across folds for a trustworthy
range before trusting any single accuracy number. (Deep-model CV is a follow-up --
too slow to run per-fold in a notebook pass; this gives the same signal cheaply.)

In [6]:
from src.evaluate import group_kfold_splits

X_all_hc, y_all_hc, groups_all = handcrafted_matrix(pd.concat([train_df, val_df, test_df], ignore_index=True))

fold_accs = []
for fold_i, (tr_idx, held_idx) in enumerate(group_kfold_splits(
        pd.concat([train_df, val_df, test_df], ignore_index=True), n_splits=3)):
    m = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                           objective="multi:softmax", num_class=3, random_state=SEED)
    m.fit(X_all_hc[tr_idx], y_all_hc[tr_idx])
    acc = (m.predict(X_all_hc[held_idx]) == y_all_hc[held_idx]).mean()
    fold_accs.append(acc)
    print(f"fold {fold_i}: held-out participants "
          f"{sorted(set(groups_all[held_idx]))}  acc={acc:.3f}")

print(f"\nleave-participant-out CV: {np.mean(fold_accs):.3f} +/- {np.std(fold_accs):.3f}")

handcrafted feats:   0%|          | 0/608 [00:00<?, ?it/s]

fold 0: held-out participants [3, 4, 8]  acc=0.885
fold 1: held-out participants [1, 7, 9]  acc=0.938
fold 2: held-out participants [2, 5, 6]  acc=0.938

leave-participant-out CV: 0.920 +/- 0.025


## 5. Deep baseline -- BiLSTM + attention

Full feature vectors (xyz + bone + velocity + visibility) with augmentation on the
train split only (2x multiplier, section 9). Val/test stay deterministic.

In [7]:
from src.data.dataset import build_feature_cache, ClipSequenceDataset

train_paths = build_feature_cache(train_df, split="train", augment=True)
val_paths = build_feature_cache(val_df, split="val", augment=False)
test_paths = build_feature_cache(test_df, split="test", augment=False)

train_ds = ClipSequenceDataset(train_paths)
val_ds = ClipSequenceDataset(val_paths)
test_ds = ClipSequenceDataset(test_paths)

sample_feat, _ = train_ds[0]
input_dim = sample_feat.shape[-1]
print(f"train: {len(train_ds)} (incl. augmented) | val: {len(val_ds)} | test: {len(test_ds)}")
from src.config import BONE_PAIRS, ANGLE_TRIPLES
expected_dim = 99 + 99 + 33 + len(BONE_PAIRS) * 3 + len(ANGLE_TRIPLES)
print(f"feature dim: {input_dim}  (expected {expected_dim} = 99 xyz + 99 vel + 33 vis + {len(BONE_PAIRS)*3} bone + {len(ANGLE_TRIPLES)} angle)")
assert input_dim == expected_dim, "feature dim mismatch -- stale cache? clear data/processed/ and rerun"

train: 1587 (incl. augmented) | val: 30 | test: 49
feature dim: 279  (expected 279 = 99 xyz + 99 vel + 33 vis + 42 bone + 6 angle)


In [8]:
from src.models.bilstm import BiLSTMBaseline
from src.train import train_baseline

model = BiLSTMBaseline(input_dim=input_dim, hidden_dim=128, num_layers=2, num_classes=3)

run_dir, best_val_acc = train_baseline(
    model, train_ds, val_ds, run_name="bilstm_baseline", device="cpu",
    extra_config={"input_dim": input_dim, "seq_len": SEQ_LEN},
)
print(f"\nbest val acc: {best_val_acc:.3f}  |  run saved to {run_dir}")

epoch   0  train_loss 0.5585 acc 0.803  val_loss 0.2219 acc 1.000
epoch   1  train_loss 0.4390 acc 0.869  val_loss 0.3720 acc 0.767
epoch   2  train_loss 0.4077 acc 0.878  val_loss 0.2642 acc 1.000
epoch   3  train_loss 0.3994 acc 0.892  val_loss 0.2221 acc 1.000
epoch   4  train_loss 0.3803 acc 0.896  val_loss 0.2260 acc 1.000
epoch   5  train_loss 0.3625 acc 0.913  val_loss 0.1765 acc 1.000
epoch   6  train_loss 0.3532 acc 0.916  val_loss 0.2349 acc 1.000
epoch   7  train_loss 0.3239 acc 0.934  val_loss 0.1813 acc 1.000
epoch   8  train_loss 0.3312 acc 0.927  val_loss 0.2125 acc 0.967
epoch   9  train_loss 0.3241 acc 0.931  val_loss 0.1821 acc 1.000
epoch  10  train_loss 0.2838 acc 0.951  val_loss 0.1959 acc 1.000
early stop at epoch 10 (no val improvement in 10 epochs)

best val acc: 1.000  |  run saved to C:\Data_Tekken\runs\bilstm_baseline_20260822_004814


## 6. Evaluate on locked test set (deterministic + TTA)

In [9]:
from src.evaluate import evaluate_dataset, predict_with_tta

model.load_state_dict(torch.load(run_dir / "best.pt"))
model.eval()

report, cm = evaluate_dataset(model, test_ds, device="cpu", class_names=ACTIONS)
print("=== BiLSTM baseline -- test (deterministic) ===")
print(pd.DataFrame(report).T)
print(cm)

C:\Users\User\AppData\Local\Temp\ipykernel_26368\618525411.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(run_dir / "best.pt"))


=== BiLSTM baseline -- test (deterministic) ===
              precision    recall  f1-score    support
kicking        1.000000  1.000000  1.000000  16.000000
punching       1.000000  0.466667  0.636364  15.000000
shooting       0.692308  1.000000  0.818182  18.000000
accuracy       0.836735  0.836735  0.836735   0.836735
macro avg      0.897436  0.822222  0.818182  49.000000
weighted avg   0.886970  0.836735  0.821892  49.000000
[[16  0  0]
 [ 0  7  8]
 [ 0  0 18]]


In [10]:

tta_preds, tta_labels = [], []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="TTA eval"):
    raw = _raw_landmarks(row)
    pred = predict_with_tta(model, raw["xyz"], raw["visibility"], seq_len=SEQ_LEN, device="cpu")
    tta_preds.append(pred)
    tta_labels.append(ACTION_TO_IDX[row["action"]])

tta_preds, tta_labels = np.array(tta_preds), np.array(tta_labels)
print("=== BiLSTM baseline -- test (TTA: orig + mirror + 2x time-warp) ===")
print(classification_report(tta_labels, tta_preds, target_names=ACTIONS))
print(confusion_matrix(tta_labels, tta_preds))

TTA eval:   0%|          | 0/49 [00:00<?, ?it/s]

=== BiLSTM baseline -- test (TTA: orig + mirror + 2x time-warp) ===
              precision    recall  f1-score   support

     kicking       1.00      1.00      1.00        16
    punching       1.00      0.47      0.64        15
    shooting       0.69      1.00      0.82        18

    accuracy                           0.84        49
   macro avg       0.90      0.82      0.82        49
weighted avg       0.89      0.84      0.82        49

[[16  0  0]
 [ 0  7  8]
 [ 0  0 18]]


## 7. Summary

Fill in after running:

- Sanity baseline (XGBoost, hand-crafted features) test accuracy: ___
- Leave-participant-out CV: ___ +/- ___
- BiLSTM baseline test accuracy (deterministic): ___
- BiLSTM baseline test accuracy (TTA): ___

**Next step:** if BiLSTM validates the pipeline (meaningfully above chance, confusion
matrix makes semantic sense), move to the 2s-AGCN fine-tune notebook per protocol
section 7, using addendum #4's discriminative fine-tuning (freeze backbone first).